# cross-entropy-classification-loss — ex3: class-weighted cross-entropy for imbalanced classes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cross-entropy-classification-loss`. Running the final beacon cell reports progress against the `Loss: Cross-entropy classification` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: Cross-entropy classification` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-entropy-classification-loss`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-entropy-classification-loss"
DD_SUBTOPIC = "Loss: Cross-entropy classification"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Class-weighted cross-entropy for imbalanced classes

Ex1 did vanilla cross-entropy. Ex2 used `ignore_index` to skip padding. The deepening move handles a different real-world case: an IMBALANCED dataset where rare classes need a heavier loss contribution.

```python
# weights[c] = how much class-c examples count. Shape (C,).
loss = F.cross_entropy(logits, labels, weight=weights)
```

**What `weight=` actually does.** PyTorch multiplies each example's per-class log-prob by `weight[label]` BEFORE averaging. The denominator of the mean is the SUM OF WEIGHTS for the labels in the batch, NOT the batch size — so the result is a weighted mean, not a sum.

```python
# Manual decomposition:
log_probs    = F.log_softmax(logits, dim=-1)       # (B, C)
per_ex       = -log_probs[range(len(labels)), labels]  # (B,) NLL
per_ex_w     = per_ex * weights[labels]            # (B,) re-weighted
loss         = per_ex_w.sum() / weights[labels].sum()  # weighted mean
```

**Common weight scheme.** `weight = 1 / class_counts` then normalized — each class contributes equally regardless of frequency. Combats class imbalance without resampling.

**Uniform weights collapse to vanilla CE.** When `weight = ones(C)`, the formula reduces to `mean(per_ex)` — exactly ex1's result. The weighted form is a strict generalization.

### Exercise 3 — class-weighted cross-entropy for imbalanced classes

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply per-class weights to cross-entropy so rare-class examples carry a heavier loss contribution, computed manually and verified against `F.cross_entropy(..., weight=...)`.
> Keywords: cross-entropy, class-weights, imbalanced, weighted-loss
> ```

**KCs targeted:** `class-weight-multiplies-per-example-loss`, `weighted-mean-divides-by-weight-sum-not-batch-size`

Implement `ex3_weighted_cross_entropy(logits, labels, weights)`. The class-weighted cross-entropy loss, computed manually.

Inputs:
- `logits`: `(B, C)` float tensor — unnormalized scores.
- `labels`: `(B,)` int64 tensor — ground-truth class indices in `[0, C)`.
- `weights`: `(C,)` float tensor — per-class loss weights (positive, but NOT necessarily normalized).

Steps:

1. `log_probs = F.log_softmax(logits, dim=-1)` — `(B, C)`.
2. Gather the log-prob of the correct class per example: `tgt_lp = log_probs[range(len(labels)), labels]` — `(B,)`.
3. Compute per-example NLL: `per_ex = -tgt_lp` — `(B,)`.
4. Gather the corresponding weights: `w_per_ex = weights[labels]` — `(B,)`.
5. Weighted mean: `(per_ex * w_per_ex).sum() / w_per_ex.sum()`.

Return a scalar tensor (0-D, `loss.shape == ()`).

**Must match `F.cross_entropy(logits, labels, weight=weights)` to floating-point tolerance.**

In [ ]:
def ex3_weighted_cross_entropy(logits, labels, weights):
    """Class-weighted cross-entropy — must match F.cross_entropy(weight=...)."""
    raise NotImplementedError()


def _test_ex3():
    # === Basic correctness against F.cross_entropy ===
    t.manual_seed(0)
    B, C = 32, 5
    logits = t.randn(B, C)
    labels = t.randint(0, C, (B,))
    weights = t.tensor([1.0, 2.0, 0.5, 4.0, 1.5])

    expected = F.cross_entropy(logits, labels, weight=weights)
    got = ex3_weighted_cross_entropy(logits, labels, weights)
    assert isinstance(got, t.Tensor), f'must return tensor, got {type(got)}'
    assert got.ndim == 0, f'must be scalar (0-D); got shape {tuple(got.shape)}'
    assert t.allclose(got, expected, atol=1e-6), (
        f'weighted CE mismatch:\n  got={got.item()}\n  expected={expected.item()}'
    )

    # === Uniform weights collapse to vanilla cross-entropy ===
    uniform = t.ones(C)
    vanilla = F.cross_entropy(logits, labels)
    got_uniform = ex3_weighted_cross_entropy(logits, labels, uniform)
    assert t.allclose(got_uniform, vanilla, atol=1e-6), (
        f'uniform weights should collapse to vanilla CE:\n  got={got_uniform.item()}\n  '
        f'vanilla={vanilla.item()}'
    )

    # === Doubling all weights does NOT change the result (weighted MEAN) ===
    got1 = ex3_weighted_cross_entropy(logits, labels, weights)
    got2 = ex3_weighted_cross_entropy(logits, labels, weights * 2.0)
    assert t.allclose(got1, got2, atol=1e-6), (
        f'scaling weights uniformly should not change weighted mean:\n  '
        f'  got1={got1.item()} got2={got2.item()}'
    )

    # === Heavier weight on a class actually changes the loss ===
    # A weight that boosts the rare-class label should pull the loss toward
    # that class's NLL contribution.
    w_uniform = t.ones(C)
    w_boost = t.ones(C); w_boost[0] = 10.0
    loss_uniform = ex3_weighted_cross_entropy(logits, labels, w_uniform)
    loss_boost = ex3_weighted_cross_entropy(logits, labels, w_boost)
    # These should be different (provided some examples have label==0).
    assert (labels == 0).any(), 'test setup: need at least one class-0 example'
    assert not t.allclose(loss_uniform, loss_boost, atol=1e-4), (
        'boosting class-0 weight should change the weighted-mean loss'
    )

    # === Imbalanced case: 1 rare example + 9 common ===
    logits_small = t.randn(10, 3)
    labels_small = t.tensor([0] * 9 + [2])  # class 2 is rare (1 of 10)
    weights_small = t.tensor([1.0, 1.0, 9.0])  # boost rare class 9x
    expected = F.cross_entropy(logits_small, labels_small, weight=weights_small)
    got = ex3_weighted_cross_entropy(logits_small, labels_small, weights_small)
    assert t.allclose(got, expected, atol=1e-6)

    # === All examples same class — weighted mean of identical entries == per-example NLL ===
    logits_one = t.randn(4, 3)
    labels_one = t.tensor([1, 1, 1, 1])
    w_one = t.tensor([0.3, 0.7, 0.2])
    got = ex3_weighted_cross_entropy(logits_one, labels_one, w_one)
    lp = F.log_softmax(logits_one, dim=-1)
    expected_manual = -lp[t.arange(4), labels_one].mean()  # weights cancel
    assert t.allclose(got, expected_manual, atol=1e-6), (
        f'when all labels are same class, weighted mean == regular mean'
    )

    # === Return type sanity ===
    result = ex3_weighted_cross_entropy(logits, labels, weights)
    assert result.dtype.is_floating_point, f'loss should be float dtype; got {result.dtype}'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_weighted_cross_entropy(logits, labels, weights):
    log_probs = F.log_softmax(logits, dim=-1)
    tgt_lp = log_probs[t.arange(len(labels)), labels]
    per_ex = -tgt_lp                       # (B,)
    w_per_ex = weights[labels]             # (B,)
    return (per_ex * w_per_ex).sum() / w_per_ex.sum()
```

**Why divide by `w_per_ex.sum()` not `B`.** PyTorch's `F.cross_entropy(weight=...)` returns a WEIGHTED MEAN — the denominator is the sum of weights for the labels appearing in the batch, not the batch size. Dividing by B would make the result scale with the weight magnitude, which is precisely what weighted-mean is designed to avoid.

**Uniform weights collapse case.** When `weights = ones(C)`, `w_per_ex = ones(B)`, `w_per_ex.sum() = B`, and the result is `per_ex.sum() / B == per_ex.mean()` — exactly ex1's vanilla cross-entropy. Strict generalization.

**Why `t.arange(len(labels))` not `range(...)`.** Both work — `F.log_softmax` returns a tensor and tensor-indexing accepts either. `t.arange` keeps everything on the same device (matters for GPU tensors); `range` would silently move the index to CPU.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()